In [ ]:
# ============================================================
# Figure 2: Identification of R-loop regulator programs
# Output directory: 01_Figure2_python
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

import matplotlib as mpl
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import NMF
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from scipy.stats import pearsonr


# ============================================================
# 0. Path config
# ============================================================

PROJECT_DIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC"

FIG1_DIR = os.path.join(PROJECT_DIR, "01_Figure1_python")
FIG2_DIR = os.path.join(PROJECT_DIR, "02_Figure2_python")
os.makedirs(FIG2_DIR, exist_ok=True)

REF_ADATA_PATH = os.path.join(
    FIG1_DIR,
    "Step10_adata_all_final_celltype.h5ad"
)

MALIGNANT_META_PATH = os.path.join(
    FIG1_DIR,
    "inferCNV_results",
    "Figure1_marker_malignant_inferCNV_intersection_metadata.csv"
)

RLOOP_GENE_FILE = os.path.join(
    PROJECT_DIR,
    "00_datacollection",
    "Rloop_regulators_Homo_sapiens.txt"
)

EXPR_ADATA_CANDIDATES = [
    os.path.join(FIG1_DIR, "Step12_high_confidence_malignant_cells.h5ad"),
    os.path.join(FIG1_DIR, "Step11_adata_with_inferCNV_major_annotation.h5ad"),
    os.path.join(FIG1_DIR, "Step11_adata_with_inferCNV_refined_subtypes.h5ad"),
    os.path.join(FIG1_DIR, "Step10_adata_all_final_celltype.h5ad"),
    os.path.join(FIG1_DIR, "Step9A_adata_all_final_annotation_raw_merged.h5ad"),
    os.path.join(FIG1_DIR, "Step5A_adata_all_raw_merged.h5ad"),
    os.path.join(FIG1_DIR, "adata_all_integrated_final.h5ad"),
]

RANDOM_STATE = 42
MIN_CELLS_PER_PSEUDOBULK = 20

N_RLOOP_MODULES = 4
N_NMF_PROGRAMS = 4
TOP_GENES_PER_NMF_PROGRAM = 50
TOP_GENES_PER_PROGRAM_PLOT = 15

MALIGNANT_CLUSTER_RESOLUTION = 0.6


# ============================================================
# 1. Plot settings
# ============================================================

mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["axes.unicode_minus"] = False


def save_fig(fig, name, w=6, h=5):
    fig.set_size_inches(w, h)

    for ext in ["png", "pdf", "svg"]:
        fig.savefig(
            os.path.join(FIG2_DIR, f"{name}.{ext}"),
            dpi=300,
            bbox_inches="tight",
            transparent=True
        )

    plt.close(fig)
    print("Saved:", name)


# ============================================================
# 2. Helper functions
# ============================================================

def read_gene_list(path):
    genes = []

    with open(path, "r") as f:
        for line in f:
            line = line.strip()

            if line == "":
                continue

            if "\t" in line:
                gene = line.split("\t")[0]
            elif "," in line:
                gene = line.split(",")[0]
            else:
                gene = line

            gene = gene.strip()

            if gene != "":
                genes.append(gene)

    return sorted(set(genes))


def match_genes(gene_list, var_names):
    var_names = pd.Index(var_names.astype(str))

    direct = [g for g in gene_list if g in var_names]

    if len(direct) >= 100:
        return sorted(set(direct))

    upper_map = {g.upper(): g for g in var_names}

    matched = sorted(set([
        upper_map[g.upper()]
        for g in gene_list
        if g.upper() in upper_map
    ]))

    return matched


def get_expr_dense(adata_obj, genes):
    genes = [g for g in genes if g in adata_obj.var_names]

    if len(genes) == 0:
        return None

    X = adata_obj[:, genes].X

    if sp.issparse(X):
        X = X.toarray()

    return np.asarray(X, dtype=np.float32)


def manual_gene_set_score(adata_obj, genes, score_name):
    genes = [g for g in genes if g in adata_obj.var_names]

    if len(genes) == 0:
        adata_obj.obs[score_name] = np.nan
        print(score_name, "genes: 0")
        return

    X = get_expr_dense(adata_obj, genes)

    if X.shape[1] == 1:
        score = X[:, 0]
    else:
        Xz = StandardScaler().fit_transform(X)
        score = Xz.mean(axis=1)

    adata_obj.obs[score_name] = score
    print(score_name, "genes:", len(genes))


# ============================================================
# 3. Load R-loop genes and Figure1 malignant metadata
# ============================================================

rloop_genes_all = read_gene_list(RLOOP_GENE_FILE)
print("Input R-loop regulators:", len(rloop_genes_all))

adata_ref = sc.read_h5ad(REF_ADATA_PATH)
meta = pd.read_csv(MALIGNANT_META_PATH, index_col=0)

common_cells = adata_ref.obs_names.intersection(meta.index)
adata_ref = adata_ref[common_cells].copy()
meta = meta.loc[common_cells].copy()

for col in meta.columns:
    adata_ref.obs[col] = meta[col].values

malignant_mask = (
    adata_ref.obs["malignant_inferCNV"].astype(str)
    == "High-confidence malignant"
)

adata_mal_ref = adata_ref[malignant_mask].copy()
malignant_barcodes = pd.Index(adata_mal_ref.obs_names)

print("High-confidence malignant cells:", len(malignant_barcodes))


# ============================================================
# 4. Select expression object with most R-loop genes
# ============================================================

candidate_records = []

for path in EXPR_ADATA_CANDIDATES:
    if not os.path.exists(path):
        continue

    ad = sc.read_h5ad(path, backed="r")
    detected = match_genes(rloop_genes_all, ad.var_names)
    overlap_mal = len(malignant_barcodes.intersection(ad.obs_names))

    candidate_records.append({
        "path": path,
        "n_obs": ad.n_obs,
        "n_vars": ad.n_vars,
        "overlap_malignant_cells": overlap_mal,
        "detected_rloop_genes": len(detected)
    })

    ad.file.close()

candidate_df = pd.DataFrame(candidate_records)

candidate_df.to_csv(
    os.path.join(FIG2_DIR, "Figure2_candidate_h5ad_gene_detection_check.csv"),
    index=False
)

print(candidate_df)

candidate_df2 = candidate_df[candidate_df["overlap_malignant_cells"] > 0].copy()

if candidate_df2.empty:
    candidate_df2 = candidate_df.copy()

candidate_df2 = candidate_df2.sort_values(
    by=["detected_rloop_genes", "n_vars", "overlap_malignant_cells"],
    ascending=False
)

EXPR_ADATA_PATH = candidate_df2.iloc[0]["path"]
print("Selected expression object:", EXPR_ADATA_PATH)

adata_expr_all = sc.read_h5ad(EXPR_ADATA_PATH)

rloop_genes = match_genes(rloop_genes_all, adata_expr_all.var_names)
print("Detected R-loop regulators:", len(rloop_genes))

pd.Series(rloop_genes).to_csv(
    os.path.join(FIG2_DIR, "Detected_Rloop_regulators.csv"),
    index=False,
    header=["gene"]
)


# ============================================================
# 5. Normalize expression object if raw count
# ============================================================

Xtmp = adata_expr_all.X
max_val = Xtmp.max() if sp.issparse(Xtmp) else np.nanmax(Xtmp)

print("Max expression value:", max_val)

if max_val > 50:
    print("Raw counts detected. Running normalize_total + log1p...")
    sc.pp.normalize_total(adata_expr_all, target_sum=1e4)
    sc.pp.log1p(adata_expr_all)
else:
    print("Expression seems already log-normalized.")


# ============================================================
# 6. Pseudo-bulk construction for module/program discovery
# ============================================================

sample_candidates = ["sample_id", "sample", "Sample", "orig.ident", "patient", "Patient"]
celltype_candidates = ["celltype", "Type", "major_celltype", "annotation"]

sample_col = None
for c in sample_candidates:
    if c in adata_expr_all.obs.columns:
        sample_col = c
        break

celltype_col = None
for c in celltype_candidates:
    if c in adata_expr_all.obs.columns:
        celltype_col = c
        break

if sample_col is None:
    raise ValueError("No sample column found.")

if celltype_col is not None:
    adata_expr_all.obs["pseudo_group_for_module"] = (
        adata_expr_all.obs[sample_col].astype(str)
        + "__"
        + adata_expr_all.obs[celltype_col].astype(str)
    )
else:
    adata_expr_all.obs["pseudo_group_for_module"] = (
        adata_expr_all.obs[sample_col].astype(str)
    )

print("Pseudo-bulk sample column:", sample_col)
print("Pseudo-bulk celltype column:", celltype_col)

X_rloop_all = get_expr_dense(adata_expr_all, rloop_genes)
groups = adata_expr_all.obs["pseudo_group_for_module"].astype(str).values

pseudo_records = []

for group in pd.unique(groups):
    idx = np.where(groups == group)[0]

    if len(idx) < MIN_CELLS_PER_PSEUDOBULK:
        continue

    mean_expr = X_rloop_all[idx, :].mean(axis=0)

    rec = {
        "pseudo_group": group,
        "n_cells": len(idx)
    }

    for i, g in enumerate(rloop_genes):
        rec[g] = mean_expr[i]

    pseudo_records.append(rec)

pseudo_module_df = pd.DataFrame(pseudo_records)

pseudo_module_df.to_csv(
    os.path.join(FIG2_DIR, "Fig2_pseudobulk_Rloop_expression.csv"),
    index=False
)

print("Pseudo-bulk matrix:", pseudo_module_df.shape)


# ============================================================
# 7. WGCNA-like clustering
# ============================================================

gene_mat = pseudo_module_df[rloop_genes].copy()
gene_mat = gene_mat.loc[:, gene_mat.var(axis=0) > 0]

rloop_genes_use = gene_mat.columns.tolist()

X_module = gene_mat.values
X_module_scaled = StandardScaler().fit_transform(X_module)

corr = np.corrcoef(X_module_scaled.T)
corr = np.nan_to_num(corr)

power = 6
adj = ((1 + corr) / 2) ** power
np.fill_diagonal(adj, 1)

dist = 1 - adj
dist = np.clip(dist, 0, 1)

link = linkage(squareform(dist, checks=False), method="average")

module_labels = fcluster(
    link,
    t=N_RLOOP_MODULES,
    criterion="maxclust"
)

module_df = pd.DataFrame({
    "gene": rloop_genes_use,
    "Rloop_module": [f"RRC{m}" for m in module_labels]
})

module_df.to_csv(
    os.path.join(FIG2_DIR, "Fig2_Rloop_gene_modules_WGCNA_like.csv"),
    index=False
)

print("RRC module sizes:")
print(module_df["Rloop_module"].value_counts())


# ============================================================
# Fig2A: WGCNA-like dendrogram
# ============================================================

fig, ax = plt.subplots(figsize=(7.2, 3.8))

dendrogram(
    link,
    no_labels=True,
    color_threshold=None,
    ax=ax
)

ax.set_title("Co-expression clustering of R-loop regulators")
ax.set_xlabel("R-loop regulators")
ax.set_ylabel("Distance")

save_fig(fig, "Fig2A_Rloop_gene_module_dendrogram", 7.2, 3.8)


# ============================================================
# Fig2B: representative RRC heatmap
# ============================================================

plot_genes = []

for mod in sorted(module_df["Rloop_module"].unique()):
    genes = module_df.loc[module_df["Rloop_module"] == mod, "gene"].tolist()
    genes = [g for g in genes if g in gene_mat.columns]

    if len(genes) == 0:
        continue

    vars_ = gene_mat[genes].var(axis=0).sort_values(ascending=False)
    plot_genes.extend(vars_.head(min(20, len(vars_))).index.tolist())

heat = gene_mat[plot_genes].copy()

heat_z = pd.DataFrame(
    StandardScaler().fit_transform(heat),
    index=heat.index,
    columns=heat.columns
)

fig, ax = plt.subplots(figsize=(10, 5.2))

im = ax.imshow(
    heat_z.T.values,
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
    aspect="auto"
)

ax.set_xticks([])
ax.set_yticks(np.arange(len(plot_genes)))
ax.set_yticklabels(plot_genes, fontsize=6)

ax.set_xlabel("Pseudo-bulk groups")
ax.set_ylabel("Representative R-loop regulators")
ax.set_title("Representative R-loop regulator modules")

cbar = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Scaled expression")

save_fig(fig, "Fig2B_Rloop_module_heatmap_pseudobulk", 10, 5.2)


# ============================================================
# 8. NMF program discovery
# ============================================================

X_nmf = MinMaxScaler().fit_transform(X_module)

nmf = NMF(
    n_components=N_NMF_PROGRAMS,
    init="nndsvda",
    random_state=RANDOM_STATE,
    max_iter=1000
)

W = nmf.fit_transform(X_nmf)
H = nmf.components_

nmf_programs = [f"RLP{i+1}" for i in range(N_NMF_PROGRAMS)]

W_df = pd.DataFrame(W, columns=nmf_programs)
W_df.insert(0, "pseudo_group", pseudo_module_df["pseudo_group"].values)

H_df = pd.DataFrame(H, columns=rloop_genes_use, index=nmf_programs)

W_df.to_csv(
    os.path.join(FIG2_DIR, "Fig2_NMF_Rloop_program_scores_pseudobulk.csv"),
    index=False
)

H_df.to_csv(
    os.path.join(FIG2_DIR, "Fig2_NMF_Rloop_program_gene_weights.csv")
)

top_records = []

for prog in nmf_programs:
    top_genes = (
        H_df.loc[prog]
        .sort_values(ascending=False)
        .head(TOP_GENES_PER_NMF_PROGRAM)
    )

    for rank, (gene, weight) in enumerate(top_genes.items(), start=1):
        top_records.append({
            "program": prog,
            "rank": rank,
            "gene": gene,
            "weight": weight
        })

nmf_top_genes_df = pd.DataFrame(top_records)

nmf_top_genes_df.to_csv(
    os.path.join(FIG2_DIR, "Fig2_NMF_top_genes_per_Rloop_program.csv"),
    index=False
)


# ============================================================
# Fig2C: NMF top gene weight heatmap
# ============================================================

nmf_plot_genes = (
    nmf_top_genes_df
    .groupby("program")
    .head(TOP_GENES_PER_PROGRAM_PLOT)["gene"]
    .drop_duplicates()
    .tolist()
)

nmf_weight_mat = H_df[nmf_plot_genes].T

nmf_weight_z = pd.DataFrame(
    StandardScaler().fit_transform(nmf_weight_mat),
    index=nmf_weight_mat.index,
    columns=nmf_weight_mat.columns
)

fig, ax = plt.subplots(figsize=(5.8, max(5, len(nmf_plot_genes) * 0.16)))

im = ax.imshow(
    nmf_weight_z.values,
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
    aspect="auto"
)

ax.set_xticks(np.arange(len(nmf_weight_z.columns)))
ax.set_xticklabels(nmf_weight_z.columns)

ax.set_yticks(np.arange(len(nmf_weight_z.index)))
ax.set_yticklabels(nmf_weight_z.index, fontsize=6)

ax.set_xlabel("NMF R-loop programs")
ax.set_ylabel("Top R-loop regulators")
ax.set_title("Top regulators defining R-loop programs")

cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("Scaled NMF weight")

save_fig(fig, "Fig2C_NMF_top_gene_weight_heatmap", 5.8, max(5, len(nmf_plot_genes) * 0.16))


# ============================================================
# 9. Prepare malignant expression object
# ============================================================

expr_cells = malignant_barcodes.intersection(adata_expr_all.obs_names)

adata_mal_expr = adata_expr_all[expr_cells].copy()
adata_mal_ref2 = adata_mal_ref[expr_cells].copy()

for col in adata_mal_ref2.obs.columns:
    adata_mal_expr.obs[col] = adata_mal_ref2.obs[col].values

for key in adata_mal_ref2.obsm.keys():
    adata_mal_expr.obsm[key] = adata_mal_ref2.obsm[key].copy()

print("Malignant expression object:")
print(adata_mal_expr)


# ============================================================
# 10. Malignant subcluster
# ============================================================

candidate_cluster_cols = [
    "malignant_subcluster",
    "malignant_cluster",
    "subcluster",
    "leiden",
    "malignant_subtypes",
    "subcluster_refined"
]

cluster_col = None

for c in candidate_cluster_cols:
    if c in adata_mal_expr.obs.columns:
        cluster_col = c
        break

if cluster_col is None:
    print("No malignant subcluster found, running Leiden...")

    if "X_pca_harmony" in adata_mal_expr.obsm:
        use_rep = "X_pca_harmony"
    elif "X_pca" in adata_mal_expr.obsm:
        use_rep = "X_pca"
    else:
        sc.pp.scale(adata_mal_expr, max_value=10)
        n_pcs = min(30, adata_mal_expr.n_vars - 1)
        sc.tl.pca(adata_mal_expr, n_comps=n_pcs, random_state=RANDOM_STATE)
        use_rep = "X_pca"

    sc.pp.neighbors(
        adata_mal_expr,
        n_neighbors=25,
        n_pcs=min(30, adata_mal_expr.obsm[use_rep].shape[1]),
        use_rep=use_rep
    )

    sc.tl.leiden(
        adata_mal_expr,
        resolution=MALIGNANT_CLUSTER_RESOLUTION,
        key_added="malignant_subcluster",
        random_state=RANDOM_STATE
    )

    cluster_col = "malignant_subcluster"

adata_mal_expr.obs[cluster_col] = adata_mal_expr.obs[cluster_col].astype(str)

print("Using malignant subcluster column:", cluster_col)
print(adata_mal_expr.obs[cluster_col].value_counts())


# ============================================================
# 11. Score RLP3 / RLP4 in malignant cells
# ============================================================

for prog in ["RLP3", "RLP4"]:
    genes = (
        nmf_top_genes_df
        .loc[nmf_top_genes_df["program"] == prog, "gene"]
        .tolist()
    )

    manual_gene_set_score(
        adata_mal_expr,
        genes,
        f"{prog}_score"
    )

core_score_mat = adata_mal_expr.obs[["RLP3_score", "RLP4_score"]].copy()

core_score_z = pd.DataFrame(
    StandardScaler().fit_transform(core_score_mat),
    index=core_score_mat.index,
    columns=core_score_mat.columns
)

adata_mal_expr.obs["RLP3_RLP4_combined_score"] = core_score_z.mean(axis=1)

median_score = adata_mal_expr.obs["RLP3_RLP4_combined_score"].median()

adata_mal_expr.obs["Rloop_program_group"] = np.where(
    adata_mal_expr.obs["RLP3_RLP4_combined_score"] >= median_score,
    "R-loop-program-high",
    "R-loop-program-low"
)

print(adata_mal_expr.obs["Rloop_program_group"].value_counts())


# ============================================================
# Fig2D: RLP activity across malignant subclusters
# ============================================================

plot_df = adata_mal_expr.obs[
    [cluster_col, "RLP3_score", "RLP4_score", "RLP3_RLP4_combined_score"]
].copy()

avg_prog = (
    plot_df
    .groupby(cluster_col)[["RLP3_score", "RLP4_score", "RLP3_RLP4_combined_score"]]
    .mean()
)

avg_prog_z = pd.DataFrame(
    StandardScaler().fit_transform(avg_prog),
    index=avg_prog.index,
    columns=avg_prog.columns
)

fig, ax = plt.subplots(figsize=(6.8, 4.5))

im = ax.imshow(
    avg_prog_z.values,
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
    aspect="auto"
)

ax.set_xticks(np.arange(len(avg_prog_z.columns)))
ax.set_xticklabels(avg_prog_z.columns, rotation=45, ha="right")

ax.set_yticks(np.arange(len(avg_prog_z.index)))
ax.set_yticklabels([f"C{x}" for x in avg_prog_z.index])

ax.set_title("NMF R-loop program activity in malignant cells")
ax.set_xlabel("R-loop program score")
ax.set_ylabel("Malignant subcluster")

for i in range(avg_prog_z.shape[0]):
    for j in range(avg_prog_z.shape[1]):
        ax.text(
            j,
            i,
            f"{avg_prog_z.iloc[i, j]:.1f}",
            ha="center",
            va="center",
            fontsize=8
        )

cbar = plt.colorbar(im, ax=ax, fraction=0.045, pad=0.02)
cbar.set_label("Scaled activity")

save_fig(fig, "Fig2D_NMF_program_activity_malignant_subclusters", 6.8, 4.5)


# ============================================================
# Fig2E: UMAP of RLP3/RLP4 scores
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.5))

for ax, score in zip(
    axes,
    ["RLP3_score", "RLP4_score", "RLP3_RLP4_combined_score"]
):
    sc.pl.umap(
        adata_mal_expr,
        color=score,
        cmap="viridis",
        ax=ax,
        frameon=False,
        show=False,
        size=8,
        title=score
    )

save_fig(fig, "Fig2E_core_Rloop_program_scores_UMAP", 10.5, 3.5)


# ============================================================
# Fig2F: R-loop program high/low UMAP
# ============================================================

fig, ax = plt.subplots(figsize=(4, 3.8))

sc.pl.umap(
    adata_mal_expr,
    color="Rloop_program_group",
    ax=ax,
    frameon=False,
    show=False,
    size=8,
    title="R-loop program group"
)

save_fig(fig, "Fig2F_Rloop_program_group_UMAP", 4, 3.8)


# ============================================================
# 12. Save final outputs
# ============================================================

adata_mal_expr.write_h5ad(
    os.path.join(FIG2_DIR, "Figure2_malignant_cells_Rloop_program_scored.h5ad")
)

adata_mal_expr.obs.to_csv(
    os.path.join(FIG2_DIR, "Figure2_malignant_cell_Rloop_program_scores_metadata.csv")
)

module_df.to_csv(
    os.path.join(FIG2_DIR, "Figure2_Rloop_gene_modules_final.csv"),
    index=False
)

nmf_top_genes_df.to_csv(
    os.path.join(FIG2_DIR, "Figure2_NMF_Rloop_program_top_genes_final.csv"),
    index=False
)

print("Figure2 completed.")
print("All outputs saved to:")
print(FIG2_DIR)